# Notebook 3: GPR foreground removal

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from meer21cm import MockSimulation
from meer21cm.plot import plot_map
from meer21cm.fg import ForegroundSimulation
from meer21cm.util import pca_clean
from gpr import *
from nautilus import Sampler, Prior
from scipy import optimize
from meer21cm.util import redshift_to_freq

In [2]:
import sys
import os
import importlib
#import functions # Replace with your actual file name
#importlib.reload(functions)
#from functions import *
from joblib import Parallel, delayed # this is for the multiprocessing but works with Jupyter notebooks
from tqdm import tqdm # this is to create a progress bar when running code 
import time

## Generate mock data

In [3]:
# Generate the mock and foreground signals
mock = MockSimulation(
    survey='meerklass_2021',
    band='L',
    flat_sky=True,
    seed=42,
    omega_hi=5e-4,
    mean_amp_1='average_hi_temp',
    tracer_bias_1=1.0,
)
fgsim = ForegroundSimulation(
    hp_nside=128,
    wproj=mock.wproj,
    num_pix_x=mock.num_pix_x,
    num_pix_y=mock.num_pix_y,
    backend='haslam',
    sp_indx_for_haslam_backend=-2.7,
)
hi_map = mock.propagate_mock_field_to_data(mock.mock_tracer_field_1)
fg_map = fgsim.fg_wcs_cube(mock.nu)


In [4]:
# extract their covariance 
cov_hi, _, _, _, = pca_clean(hi_map,1,weights=mock.w_HI,return_analysis=True,mean_center=True)
cov_fg, _, _, _, = pca_clean(fg_map,1,weights=mock.w_HI,return_analysis=True,mean_center=True)

### Estimates

In [6]:
# test a rough estimate of the amplitude of the cov
sigma_hi_kern = np.sqrt(np.diagonal(cov_hi).mean())
l_hi = 1
sigma_fg_kern = np.sqrt(np.diagonal(cov_fg).mean())* 1.0
l_fg = 20

In [ ]:
[1.90448375e-04 1.82551820e-01 1.00000000e+00 4.00000000e+02]